# Using the eeg_denoising library

## Import Dependencies

In [ ]:
from eeg_denoising import montage
from eeg_denoising.eeg_utils import load_edf, iter_edfs, get_segments, extract_signal, extract_all_signals, apply_annotations
from eeg_denoising.data import load_master, load_patient, iter_patients
from eeg_denoising.config import PATH_TO_TUAR_EDF_FILES, PATH_TO_TUAR_JSON
from pathlib import Path

## Load metadata from master.json file

In [ ]:
master = load_master()
print(f"total patients:  {len(master['Patients'])}")
print(f"total edf files: {master['summary']['total_edf_files']}")
print(f"total duration:  {master['summary']['total_duration'] / 3600:.1f} hours")
print(f"labels:          {list(master['summary']['label_totals'].keys())}")

## Load metadata for a single patient by id

In [ ]:
patient = load_patient("aaaaaaju")

print(f"patient id:    {patient['id']}")
print(f"has seizure:   {patient['has_seiz']}")
print(f"recordings:    {list(patient['recordings'].keys())}")
print(f"labels:        {list(patient['labels'].keys())}")

## Load a single recording

In [ ]:
recording = patient["recordings"]["aaaaaaju_s005_t000"]

data, ch_names, sfreq = load_edf(recording, PATH_TO_TUAR_EDF_FILES, montage.TUAR_TCP_AR)

print(f"shape:    {data.shape}")
print(f"channels: {ch_names}")
print(f"sfreq:    {sfreq} Hz")
print(f"duration: {data.shape[1] / sfreq:.1f}s")

## Inspect labels for a recording

In [ ]:
print("labels:     ", set(recording["labels"].keys()))
print("seiz labels:", set(recording["seiz_labels"].keys()))
print("has seizure:", recording["has_seiz"])

## Get A Specific Segment

In [ ]:
all_segments  = get_segments(recording)
eyem_segments = get_segments(recording, labels="eyem")
targeted      = get_segments(recording, labels="eyem", channels="FP1-F7")
with_seiz     = get_segments(recording, include_seiz=True)

print(f"all:      {len(all_segments)}")
print(f"eyem:     {len(eyem_segments)}")
print(f"targeted: {len(targeted)}")
print(f"with seiz:{len(with_seiz)}")

## Extract a single signal

In [ ]:
seg    = eyem_segments[0]
signal = extract_signal(data, ch_names, seg, sfreq)

print(f"label:    {seg['label']}")
print(f"channel:  {seg['channel']}")
print(f"start:    {seg['start']:.2f}s")
print(f"duration: {seg['duration']:.2f}s")
print(f"shape:    {signal.shape}")

## Build X and y for ML

In [ ]:
X, y = extract_all_signals(data, ch_names, sfreq, recording, include_seiz=True)

print(f"segments:     {len(X)}")
print(f"labels:       {set(y)}")
print(f"sample shape: {X[0].shape}")

## Iterate all recordings for a patient

In [ ]:
for recording_id, data, ch_names, sfreq, recording in iter_edfs(patient, PATH_TO_TUAR_EDF_FILES, montage.TUAR_TCP_AR):
    X, y = extract_all_signals(data, ch_names, sfreq, recording, include_seiz=True)
    print(f"{recording_id}: {data.shape} — {len(X)} segments — labels: {set(y)}")

## Visualise with annotations using MNE

In [ ]:
import mne

fpath = PATH_TO_TUAR_EDF_FILES / Path(recording["edf"]).name
raw   = mne.io.read_raw_edf(fpath, preload=True)

apply_annotations(raw, recording, include_seiz=True)

raw.plot(duration=10, n_channels=22, scalings="auto", block=True)